<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Classification_gpt-4o_Five Pass_POST_PROCESSING v4_FINAL**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> # **Version 7.5.6 Changes**
>   - **Five-Pass Classification**:
>   - Pass 1: Initial LLM classification (attribute-only, full context)
>   - Pass 2: Self-consistency check — LLM reviews its own justification vs classification and corrects mismatches
>   - Pass 3: Manager→Director promotion check for strategic/executive scope
>   - Pass 4: Technician→Skilled Laborer review for physical trade-based work
>   - Pass 5: Manager, Coordinator, Coach review for reclassification
> - **Addresses Technician vs Skilled Laborer confusion**
> - **Addresses Specialist underusage**
> - implements changes—especially the improved discourage_specialist function and the enhanced fifth-pass prompt
> - **Preserves all v7.5.4 logic**: Two-pass, Director pass, validation
> - **Still ignores original job title**, uses full MNPS role/competency context
> - **Same GPT-4o-2024-11-20 model**, rate-limiting, and output saving

# ==== 1) Imports, paths, inputs from v7.1 artifacts ====

In [ ]:
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI
# Mount Google Drive
drive.mount('/content/drive')
# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)
# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")
# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')
# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")
# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"
# If main input file not found, check inside the zip
if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")
    # Optionally extract Ground Truth if present
    if "Ground Truth Masterfile.csv" in zip_contents:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)
            print("✅ Extracted Ground Truth Masterfile.csv from zip")
# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"
print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")
# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')
print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

Mounted at /content/drive
📁 Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326
📁 Outputs dir: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs
📦 Found /content/MNPS Prompt Resources.zip, extracting...
✅ Extracted MNPS Prompt Resources
📄 Batch input: /content/Sample JDs.csv
📄 Ground truth: /content/Ground Truth Masterfile.csv
📄 MNPS roles: /content/MNPS Roles.csv
📄 MNPS KSACs: /content/MNPS KSACs.csv
📄 Competency Extended: /content/Competency Extended Descriptions.csv
📄 Korn Ferry: /content/Korn_Ferry Lominger 38 Competencies.csv
✅ Loaded 102 job descriptions
✅ Loaded 176 ground truth records
✅ Loaded 63 MNPS roles
✅ Loaded 315 MNPS KSACs
✅ Loaded 38 competency descriptions
✅ Loaded 38 Korn Ferry competencies


# ==== 2) Load data and build attribute-only view (ignore title) ====

In [ ]:
# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()
# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]
# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']
print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")

✅ Built attribute-only view for 102 job descriptions
✅ Ignoring job titles - focusing on job attributes only


# ==== 3) Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic ====

In [ ]:
# Get MNPS roles from the loaded data
# Handle different possible column names
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    # Fallback to first column
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()
print(f"✅ Found {len(VALID_ROLES)} MNPS roles")
# Enhanced closed sets for major and minor role groups
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator', 'Skilled Laborer',
    'Administrative Assistant'
]
MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']
# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}
# Enhanced specialist fallback patterns with Problem Role Cheat Sheet logic
# Updated to be more conservative and context-aware to prevent over-classification of Specialist roles
SPECIALIST_FALLBACKS = [
    # Teacher patterns - classroom instruction, curriculum, students (Strong signal)
    ('Teacher', r'\b(teacher|classroom|lesson|instruction|teaching|curriculum|students|educational|academic|teach)\b'),
    # Coach patterns - instructional support, mentoring, professional development for staff (Strong signal)
    ('Coach', r'\b(coach|instructional coach|plc|model lessons?|co-teach|mentor|professional development|instructional support|facilitates professional learning)\b'),
    # Analyst patterns - data analysis, research, evaluation, statistical work (Strong signal)
    ('Analyst', r'\b(analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports|data collection|data reporting)\b'),
    # Technician patterns - hands-on technical work, equipment maintenance/troubleshooting (Strong signal)
    ('Technician', r'\b(technical|repair|troubleshoot|install|equipment|hands-on|hardware|software|systems|diagnose|fix|maintain|AV support|IT support)\b'),
    # Accountant patterns - financial, accounting, bookkeeping, payroll (Strong signal)
    ('Accountant', r'\b(accounting|financial|bookkeeping|audit|payroll|finance|accounts payable|accounts receivable|fiscal|budget tracking|compensation analysis)\b'),
    # Architect patterns - building/construction vs technology (Strong signal if primary focus)
    ('Architect (Facility-Focused)', r'\b(building|construction|facility|architectural|design|space planning|renovation|infrastructure|project management|capital improvement)\b'),
    ('Architect (Technology-Focused)', r'\b(system|software|technology|IT|database|network|programming|technical architecture)\b'),
    # Counselor patterns - guidance, therapy, mental health (Strong signal)
    ('Counselor', r'\b(counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological|student services)\b'),
    # Clerical Support patterns - administrative, office work, records (Strong signal)
    ('Clerical Support', r'\b(clerk|clerical|records|data entry|office support|administrative|filing|correspondence|reception|scheduling)\b'),
]
# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']
def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')
def discourage_specialist(text: str, proposed_major: str) -> str:
    """
    Discourage 'Specialist' only when there is strong evidence for a more specific role.
    Requires multiple keywords or role-defining language to prevent over-correction.
    """
    if proposed_major != 'Specialist':
        return proposed_major

    t = (text or '').lower()

    # Only reclassify if there's STRONG evidence of a better fit
    # Check for TEACHER (direct instruction)
    if any(kw in t for kw in ['teach', 'instruction', 'lesson', 'curriculum', 'classroom']):
        return 'Teacher'

    # Check for COACH (instructional adult learning)
    if 'instructional' in t and any(kw in t for kw in ['coach', 'mentor', 'plc', 'co-teach', 'professional development']):
        return 'Coach'

    # Check for ANALYST (statistical/data analysis focus)
    if any(kw in t for kw in ['statistical', 'quantitative analysis', 'regression', 'modeling']) or \
       (t.count('analyze') >= 2 and 'data' in t):
        return 'Analyst'

    # Check for TECHNICIAN (hands-on hardware/systems)
    if any(kw in t for kw in ['troubleshoot hardware', 'repair equipment', 'install systems']):
        return 'Technician'

    # Check for ACCOUNTANT (financial accounting tasks)
    if any(kw in t for kw in ['bookkeeping', 'audit', 'accounts payable', 'general ledger']):
        return 'Accountant'

    # Otherwise, keep as Specialist
    return 'Specialist'
def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    """Distinguish between Supervisor and Manager based on education requirements.
    Supervisor: Primarily manages people, no post-high school education required
    Manager: Does more than manage people, requires minimum associates degree
    """
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Check for education requirements
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)
    # Check for broader responsibilities beyond people management
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)
    # If has degree requirement or broader responsibilities, likely Manager
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'
    # If primarily people management without degree requirements, likely Supervisor
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'
    return proposed_major
def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    """Refine distinctions between Coordinator, Coach, and Manager based on Problem Role Cheat Sheet."""
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Coach patterns - instructional support, mentoring, professional development
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'
    # Manager patterns - strategic planning, policy, budget, supervision
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'
    # Coordinator patterns - coordination, organization, facilitation
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'
    return proposed_major
def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    """Fix minor sub-grouping for executive roles - rarely "Lead", usually "I", "II", or "III"."""
    if major_role not in EXECUTIVE_ROLES:
        return minor_role
    # If it's an executive role and currently "Lead", downgrade to "III" or "II"
    if minor_role == 'Lead':
        # Check if it's a very senior executive role that might warrant "III"
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'
    return minor_role
print("✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined")

✅ Found 63 MNPS roles
✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined


# ==== 4) Build comprehensive KSACs text from all MNPS resources ====

In [ ]:
def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n"
    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()
    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)
    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")
    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)
    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")
    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)
    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")
    return ksacs_text
KSACS_TEXT = build_ksacs_text()
print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")

✅ Built comprehensive KSACs text (37793 characters)
✅ Includes all 4 critical MNPS resource documents


# ==== 5) Enhanced Zero Shot Prompt with Problem Role Cheat Sheet Guidelines ====

In [ ]:
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.
Process:
- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Group jobs that have similar functions, responsibilities, and requirements, regardless of their job titles.
- Use the attached reference sources (Ground Truth Masterfile, MNPS Roles, MNPS KSACs) to ensure alignment with MNPS standards and classifications.
- Focus on the actual work being performed, not the job title, to create meaningful and accurate groupings.
- Ensure that each grouping reflects the true nature of the work and aligns with MNPS role classifications and competency frameworks.
- Provide clear justification for each grouping decision based on the job attributes and MNPS standards.
- Never justify classifications based on job titles - only use job attributes and MNPS standards.
IMPORTANT CLASSIFICATION GUIDELINES (Based on Problem Role Cheat Sheet):
ROLE DISTINCTIONS:
- **Technician vs Specialist vs Analyst**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting
  * Specialist: Specialized knowledge in specific domain, but prefer more specific roles when possible
  * Analyst: Data analysis, research, evaluation, assessment, statistical work, reporting
- **Coordinator vs Coach vs Manager**:
  * Coordinator: Coordination, organization, facilitation, liaison work, program coordination
  * Coach: Instructional support, mentoring, professional development, co-teaching, PLC facilitation
  * Manager: Strategic planning, policy development, budget oversight, supervision, management
- **Supervisor vs Manager**:
  * Supervisor: Primarily manages people, no post-high school education required
  * Manager: Does more than manage people, requires minimum associates degree
- **Architect Roles**:
  * Architect (Facility-Focused): Building/construction/space planning/renovation/infrastructure
  * Architect (Technology-Focused): System/software/IT/database/network/programming
MINOR SUB-GROUP GUIDELINES:
- **Executive Roles** (Coordinator, Principal, Director, Manager): Rarely "Lead", usually "I", "II", or "III"
- **"Lead"** should be reserved for non-executive roles that lead teams or projects
- **"III"** for very advanced KSACs and senior-level expertise
- **"II"** for intermediate complexity and responsibility
- **"I"** for entry-level or basic complexity
Output Requirements:
- Major Role Group: Choose from approved MNPS major role groupings
- Minor Sub Group: Use I, II, III, or Lead based on complexity and responsibility level (consider executive role guidelines)
- new_job_title: Should incorporate both major_role_group and minor_sub_group (e.g., "Accountant II", "Facility Coordinator II")
- Provide detailed justification based on job attributes and MNPS KSACs alignment that matches your selected role and level"""
print("✅ Enhanced zero shot prompt with Problem Role Cheat Sheet guidelines defined")
# NEW: Self-Consistency Prompt for Pass 2
self_consistency_prompt = \
"""You previously classified a job description and provided a justification. Now, review your own output for internal consistency.
**Instructions:**
- Compare your stated `major_role_group`, `minor_sub_group`, and `new_job_title` with your `grouping_justification`.
- If the justification **does not logically support** the selected role or level, **correct the classification** to match the reasoning.
- If the justification **supports a different role** (e.g., justification describes coaching but role is "Coordinator"), update the role accordingly.
- **Do not change the justification**—only update the classification fields if they conflict with it.
- Use **only approved MNPS roles** and **minor levels (I, II, III, Lead)**.
- Ensure `new_job_title` reflects the corrected role and level.
- If already consistent, return the original values unchanged.
**Return your response as a JSON object with this exact structure:**
{
  "new_job_title": "...",
  "major_role_group": "...",
  "minor_sub_group": "...",
  "grouping_justification": "..."  // <-- DO NOT MODIFY THIS FIELD
}"""
print("✅ Self-consistency prompt for Pass 2 defined")
# NEW: Manager→Director Promotion Check Prompt for Pass 3
triple_check_prompt = """
You are performing a third-pass "promotion check" for MNPS job classification.
You will ONLY be asked to review roles that were previously classified as "Manager".
Your job is to decide whether the role should remain "Manager" or be elevated to "Director".
Use these MNPS-specific guidelines:
1. Scope of responsibility
   - Manager: Owns a program, team, or sub-unit within an office/department; scope is usually local or departmental.
   - Director: Owns an entire function, office, or district-wide program area (often multiple programs) with system-wide impact.
2. Strategy vs operations
   - Manager: Focuses on implementation, day-to-day operations, and executing strategy set by others.
   - Director: Develops or co-develops strategy, sets direction and priorities, and is responsible for long-term planning.
3. People leadership
   - Manager: Supervises individuals and small teams (specialists, coordinators, technicians).
   - Director: Leads managers and/or multiple teams; provides leadership for an office or functional area.
4. Decision rights, budget, and policy
   - Manager: Implements policies and manages part of a budget within limits set by others.
   - Director: Develops or significantly shapes policies and procedures; owns or co-owns budgets and resource allocation for their function.
5. Accountability and stakeholders
   - Manager: Accountable for performance of a team/program; works mainly with school staff, principals, and department peers.
   - Director: Accountable for district- or system-level outcomes; collaborates with Chiefs, Executive Leadership, and external agencies; often represents MNPS externally.
Upgrade to "Director" ONLY IF the job description clearly shows MOST of the Director characteristics above,
such as district-wide scope, strategy setting, policy/budget ownership, and leadership of other leaders or multiple teams.
If evidence is mixed or ambiguous, KEEP IT AS "Manager".
Important constraints:
- Ignore the original job title text; use duties, responsibilities, scope, and KSAC-related content.
- You may ONLY choose "Manager" or "Director" as major_role_group.
- Keep the minor_sub_group consistent with MNPS conventions (I, II, III, or Lead; executive roles rarely have "Lead").
Return a JSON object with:
{
  "new_job_title": "Updated descriptive title if you upgrade to Director, otherwise keep or lightly refine",
  "major_role_group": "Manager" or "Director",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Explain why you kept Manager or upgraded to Director, citing specific phrases from the job description."
}
"""
print("✅ Manager→Director promotion check prompt for Pass 3 defined")
# NEW: Technician→Skilled Laborer Review Prompt for Pass 4
skilled_laborer_check_prompt = """
You are performing a fourth-pass "Technician vs Skilled Laborer" review for MNPS job classification.
You will ONLY be asked to review roles that were previously classified as "Technician".
Your job is to decide whether the role should remain "Technician" or be reclassified as "Skilled Laborer".
Use these MNPS-specific guidelines:
**Technician:**
- Focus on technical systems work requiring diagnostics and specialized technical knowledge
- HVAC systems (heating, ventilation, air conditioning) - technical diagnostics and system troubleshooting
- Network infrastructure, IT systems, audio-visual systems
- Automotive/vehicle diagnostics and repair
- Behavioral health interventions (e.g., Registered Behavior Technician with specialized training in ABA)
- Equipment troubleshooting requiring technical diagnostics and specialized certifications
- Installation and configuration of complex technical systems
- Examples: HVAC Technician, IT Technician, Audio-Visual Technician, Automotive Technician, Behavior Technician
**Skilled Laborer:**
- Focus on traditional trade-based work (plumbing, electrical, carpentry, painting, grounds)
- Physical work using hand tools and power tools to build, move, repair, or maintain physical environment
- Installation, assembly, and repair of physical structures and fixtures (not complex technical systems)
- Examples of trade work: plumbing (pipes, fixtures, drains), electrical wiring and fixtures, carpentry (building/repairing structures), painting, grounds maintenance, furniture assembly/moving, light construction
- May require trade skills and certifications but emphasizes hands-on physical work over technical diagnostics
**Key Distinction:**
- **Technician** = Technical diagnostics, troubleshooting systems, specialized technical knowledge (HVAC diagnostics, IT systems, behavioral interventions)
- **Skilled Laborer** = Traditional trades with physical hands-on work (plumbing, electrical work, carpentry, painting, grounds maintenance)
**Decision Rules:**
- Plumbing work (installing pipes, fixtures, drains) → Reclassify as "Skilled Laborer"
- Electrical work (wiring, installing fixtures) → Reclassify as "Skilled Laborer"
- Carpentry (building, repairing structures) → Reclassify as "Skilled Laborer"
- HVAC systems (technical diagnostics, system troubleshooting) → Keep as "Technician"
- IT/Network systems → Keep as "Technician"
- Behavioral interventions (RBT, ABA) → Keep as "Technician"
- If evidence is mixed or unclear → Keep as "Technician"
Important constraints:
- Ignore the original job title text; use duties, responsibilities, and work activities.
- You may ONLY choose "Technician" or "Skilled Laborer" as major_role_group.
- Keep the same minor_sub_group (I, II, III, or Lead) unless there is a strong reason to change it.
Return a JSON object with:
{
  "new_job_title": "Updated title if you reclassify to Skilled Laborer, otherwise keep",
  "major_role_group": "Technician" or "Skilled Laborer",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Explain why you kept Technician or reclassified to Skilled Laborer, citing specific work activities from the job description."
}
"""
print("✅ Technician→Skilled Laborer review prompt for Pass 4 defined")

✅ Enhanced zero shot prompt with Problem Role Cheat Sheet guidelines defined
✅ Self-consistency prompt for Pass 2 defined
✅ Manager→Director promotion check prompt for Pass 3 defined
✅ Technician→Skilled Laborer review prompt for Pass 4 defined


# ==== 6) OpenAI API Setup with Rate Limiting Protection ====

In [ ]:
import os
from google.colab import userdata
# Get API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Initialize OpenAI client
client = OpenAI()
# Use GPT-4o-2024-11-20 for stable performance
MODEL_ID = "gpt-4o-2024-11-20"
print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")
def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            error_str = str(e).lower()
            # Check for rate limiting errors
            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    # Exponential backoff with jitter
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting. Error: {e}")
                    raise e
            else:
                # Non-rate limiting error, raise immediately
                print(f"❌ Non-rate limiting error: {e}")
                raise e
    # This should never be reached, but just in case
    raise Exception("Unexpected error in retry logic")
print("✅ call_llm_json_with_retry function defined with rate limiting protection")

✅ OpenAI client initialized
✅ Using model: gpt-4o-2024-11-20
✅ call_llm_json_with_retry function defined with rate limiting protection


# ==== 7) Five-Pass Batch Processing with Self-Consistency + Manager→Director + Technician→Skilled Laborer + Coach/Coord/Manager ====

In [ ]:
from tqdm import tqdm
def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description using:
       - Pass 1: Initial classification
       - Pass 2: Self-consistency check
       - Pass 3: Manager→Director promotion check (only if final_major == 'Manager')
       - Pass 4: Technician→Skilled Laborer review (only if final_major == 'Technician')"""
    # Build job description text (ignore job title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""
    # === PASS 1: Initial Classification ===
    pass1_prompt = f"""{zero_shot_prompt}
Available MNPS Roles: {', '.join(VALID_ROLES)}
{KSACS_TEXT}
Job Description to Classify:
{job_text}
**IMPORTANT**:
- Ignore the job title completely
- Base classification solely on job attributes
- Use only approved MNPS roles and levels (I, II, III, Lead)
- Apply Problem Role Cheat Sheet guidelines
- Avoid overusing "Specialist"
- Distinguish Supervisor vs Manager based on education requirements
- Use Architect (Facility-Focused) for building/construction roles
- Use Architect (Technology-Focused) for system/software roles
- Executive roles (Coordinator, Principal, Director, Manager) rarely have "Lead" minor sub-grouping
- Ensure new_job_title incorporates both major_role_group and minor_sub_group
- Provide detailed justification that aligns with your selected role and level
Return your response as a JSON object with the following structure:
{{
  "new_job_title": "Descriptive title incorporating major_role_group and minor_sub_group",
  "major_role_group": "One of the approved MNPS roles",
  "minor_sub_group": "I, II, III, or Lead (consider executive role guidelines)",
  "grouping_justification": "Detailed explanation based on job attributes and KSACs alignment that matches your selected role and level"
}}"""
    try:
        # Get initial prediction
        pass1_response = call_llm_json_with_retry(pass1_prompt, MODEL_ID)
        # Apply post-processing logic (as in original)
        major_role = pass1_response.get('major_role_group', 'Other')
        minor_role = pass1_response.get('minor_sub_group', 'I')
        justification = pass1_response.get('grouping_justification', 'No justification provided')
        new_job_title = pass1_response.get('new_job_title', f"{major_role} {minor_role}")
        # Apply rule-based corrections
        major_role = discourage_specialist(job_text, major_role)
        major_role = distinguish_supervisor_manager(job_text, major_role)
        major_role = refine_coordinator_coach_manager(job_text, major_role)
        minor_role = normalize_minor(minor_role)
        minor_role = fix_executive_minor_sub_grouping(major_role, minor_role)
        if not new_job_title or new_job_title == 'Unknown':
            new_job_title = f"{major_role} {minor_role}"
        # Reconstruct clean Pass 1 output
        pass1_clean = {
            "new_job_title": new_job_title,
            "major_role_group": major_role,
            "minor_sub_group": minor_role,
            "grouping_justification": justification
        }
        # === PASS 2: Self-Consistency Check ===
        pass2_prompt = f"""{self_consistency_prompt}
**Your Previous Output:**
{json.dumps(pass1_clean, indent=2)}
**Now perform the self-consistency check and return the corrected (or unchanged) JSON.**
"""
        # Call LLM again for consistency check
        pass2_response = call_llm_json_with_retry(pass2_prompt, MODEL_ID)
        # Extract final values (do NOT re-apply post-processing to avoid overriding LLM correction)
        final_major = pass2_response.get('major_role_group', major_role)
        final_minor = pass2_response.get('minor_sub_group', minor_role)
        final_title = pass2_response.get('new_job_title', new_job_title)
        final_justification = pass2_response.get('grouping_justification', justification)  # should be unchanged
        # Re-normalize minor role (in case LLM outputs "1", etc.)
        final_minor = normalize_minor(final_minor)
        # === PASS 3: Manager → Director Triple-Check (ONLY for Manager) ===
        final_final_major = final_major
        final_final_minor = final_minor
        final_final_title = final_title
        final_final_justification = final_justification
        if final_major == "Manager":
            pass3_input = {
                "new_job_title": final_title,
                "major_role_group": final_major,
                "minor_sub_group": final_minor,
                "grouping_justification": final_justification
            }
            pass3_prompt = f"""{triple_check_prompt}
Full Job Description (no title):
{job_text}
Previous Classification (after two-pass check):
{json.dumps(pass3_input, indent=2)}
Now apply the promotion check and return the corrected (or unchanged) JSON.
Remember:
- You may only choose "Manager" or "Director" as major_role_group.
- Upgrade to Director ONLY with strong evidence of Director-level scope, strategy, policy/budget ownership, and leadership of other leaders or multiple teams.
- If evidence is mixed or unclear, keep Manager.
"""
            try:
                pass3_response = call_llm_json_with_retry(pass3_prompt, MODEL_ID)
                proposed_major = pass3_response.get('major_role_group', final_major)
                # Only accept Manager/Director; ignore any other surprise roles
                if proposed_major in ["Manager", "Director"]:
                    final_final_major = proposed_major
                    final_final_minor = normalize_minor(
                        pass3_response.get('minor_sub_group', final_minor)
                    )
                    final_final_title = pass3_response.get('new_job_title', final_title) or final_title
                    final_final_justification = pass3_response.get(
                        'grouping_justification', final_final_justification
                    )
                # else: silently keep Manager if model misbehaves
            except Exception as e3:
                # If triple-check fails, just keep the two-pass result
                print(f"Warning: Pass 3 failed for row {row_idx}: {e3}")
        # === PASS 4: Technician → Skilled Laborer Review (ONLY for Technician) ===
        if final_final_major == "Technician":
            pass4_input = {
                "new_job_title": final_final_title,
                "major_role_group": final_final_major,
                "minor_sub_group": final_final_minor,
                "grouping_justification": final_final_justification
            }
            pass4_prompt = f"""{skilled_laborer_check_prompt}
Full Job Description (no title):
{job_text}
Previous Classification (after previous passes):
{json.dumps(pass4_input, indent=2)}
Now apply the Technician vs Skilled Laborer review and return the corrected (or unchanged) JSON.
Remember:
- You may only choose "Technician" or "Skilled Laborer" as major_role_group.
- Reclassify to Skilled Laborer ONLY if the role primarily involves physical, trade-based work with hand/power tools.
- If the role involves technical systems work, diagnostics, or specialized technical knowledge, keep Technician.
- If evidence is mixed or unclear, keep Technician.
"""
            try:
                pass4_response = call_llm_json_with_retry(pass4_prompt, MODEL_ID)
                proposed_major = pass4_response.get('major_role_group', final_final_major)
                # Only accept Technician/Skilled Laborer; ignore any other surprise roles
                if proposed_major in ["Technician", "Skilled Laborer"]:
                    final_final_major = proposed_major
                    final_final_minor = normalize_minor(
                        pass4_response.get('minor_sub_group', final_final_minor)
                    )
                    final_final_title = pass4_response.get('new_job_title', final_final_title) or final_final_title
                    final_final_justification = pass4_response.get(
                        'grouping_justification', final_final_justification
                    )
                # else: silently keep Technician if model misbehaves
            except Exception as e4:
                # If fourth pass fails, just keep the previous result
                print(f"Warning: Pass 4 failed for row {row_idx}: {e4}")
        # Return final result (after optional Pass 3 and Pass 4)
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': final_final_title,
            'major_role_group': final_final_major,
            'minor_sub_group': final_final_minor,
            'grouping_justification': final_final_justification,
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }
# Process all job descriptions with rate limiting protection
results = []
print("🚀 Starting FOUR-pass batch processing with Self-Consistency + Director + Skilled Laborer...")
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    # Add small delay between requests to prevent rate limiting
    time.sleep(0.2)
# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v754_four_pass.csv"
results_df.to_csv(output_path, index=False)
print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")

🚀 Starting FOUR-pass batch processing with Self-Consistency + Director + Skilled Laborer...


Processing jobs: 100%|██████████| 102/102 [19:24<00:00, 11.42s/it]

✅ Processed 102 job descriptions
✅ Saved results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs/Job_Classifications_Batch_gpt4o_v754_four_pass.csv


# ==== 7B) Fifth-Pass Coach / Coordinator / Manager Refinement ====

In [ ]:
import math
FIFTH_PASS_PROMPT = """
You are performing a very narrow refinement of an existing job classification in the MNPS job classification system.
The role has already been classified as one of: "Coach", "Coordinator", or "Manager".
Your job is to decide whether to KEEP that label or CHANGE it to a better one among those three, using the definitions and rules below.

**Definitions:**
- **Coach (non-athletic):**
  - *Focus:* Improving instructional practice and adult learning (teachers, school staff, leaders).
  - *Typical Actions:* Models instruction, co-teaches, facilitates professional learning, coaching cycles, classroom observations, feedback on instruction.
  - *Authority:* No formal staff supervision or budget ownership.
- **Coordinator:**
  - *Focus:* Coordinating programs, processes, logistics, or services.
  - *Typical Actions:* Coordinates activities, schedules, maintains records, tracks data, monitors compliance, organizes events or services.
  - *Authority:* May give work direction, but usually does not formally supervise staff or own budgets. Often "monitors" or "assists" with budgets.
- **Manager:**
  - *Focus:* Leading a function or team with formal accountability.
  - *Typical Actions:* Plans, organizes, and directs a team or department; supervises and evaluates staff; sets goals; develops and manages budgets; approves expenditures; allocates resources.
  - *Authority:* Clear responsibility for staff supervision and/or budget ownership, and overall performance of a function.

**Refinement Rules (Apply these decisively):**
1. **Supervision/Budget Ownership = Manager:**
   - If the job description mentions supervising staff (hires, evaluates, disciplines) OR managing/developing a budget, classify as **Manager**, regardless of other duties.
2. **Education Requirement Rule:**
   - If the Education requirement includes "Master's Degree" OR "Administrative License," the role is almost certainly **Manager**, not Coordinator or Coach.
3. **Instructional Coaching = Coach:**
   - If most essential functions describe instructional coaching, modeling practice, classroom observations, lesson planning support, or professional learning for staff, and there is NO staff/budget authority, prefer **Coach**.
4. **Program Logistics = Coordinator:**
   - If the work is primarily about coordinating programs, services, schedules, data, or compliance, and there is limited/no formal staff/budget authority, prefer **Coordinator**.

**Conservatism rule:**
- Only keep the original label if you are confident it is correct after applying the rules above.

**You MUST ignore the original job title text and focus on duties, responsibilities, scope, and authority.**

Return your answer as a single JSON object with this exact structure:
{
  "refined_role": "Coach" | "Coordinator" | "Manager",
  "change_decision": "keep" | "change",
  "reason": "Short explanation (1–3 sentences) citing the main evidence."
}
"""
ALLOWED_COACH_ROLES = ["Coach", "Coordinator", "Manager"]
def build_job_text_from_index(idx: int) -> str:
    """
    Rebuild the attribute-only job text for a given source_row_index
    using the same ATTR_COLS as earlier in the notebook.
    """
    try:
        base_row = df.loc[idx]
    except Exception:
        # Fallback: if index alignment changed, try iloc
        try:
            base_row = df.iloc[int(idx)]
        except Exception:
            return ""
    pieces = []
    for col in ATTR_COLS:
        if col in base_row:
            val = base_row[col]
            if isinstance(val, float) and math.isnan(val):
                continue
            pieces.append(str(val))
    return " ".join(pieces)
def refine_manager_coach_coordinator_pass5(row) -> dict:
    """
    Apply the fifth-pass refinement ONLY to roles currently classified
    as Coach, Coordinator, or Manager.
    Returns a dict with the possibly-updated major_role_group and an optional reason.
    """
    prior_label = row.get("major_role_group", "")
    if prior_label not in ALLOWED_COACH_ROLES:
        # Do not change anything outside these three roles
        return {
            "major_role_group": prior_label,
            "reason_pass5": ""
        }
    source_idx = row.get("source_row_index", None)
    job_text = build_job_text_from_index(source_idx) if source_idx is not None else ""
    prompt = f"""{FIFTH_PASS_PROMPT}
Existing classification: {prior_label}
Full Job Description (no title):
{job_text}
"""
    try:
        resp = call_llm_json_with_retry(prompt, MODEL_ID)
    except Exception as e:
        print(f"⚠️ Fifth pass failed for source_row_index={source_idx}: {e}")
        # On any error, keep the original label
        return {
            "major_role_group": prior_label,
            "reason_pass5": f"Fifth pass error, kept original label: {e}"
        }
    refined_role = resp.get("refined_role", prior_label)
    change_decision = str(resp.get("change_decision", "keep")).strip().lower()
    reason = resp.get("reason", "")
    # Validate refined_role
    if refined_role not in ALLOWED_COACH_ROLES:
        refined_role = prior_label
    # Apply conservatism rule: only change if explicitly told to change
    if not change_decision.startswith("change"):
        refined_role = prior_label
    return {
        "major_role_group": refined_role,
        "reason_pass5": reason
    }
print("🚀 Starting Fifth-Pass refinement for Coach / Coordinator / Manager...")
refined_records = results_df.apply(refine_manager_coach_coordinator_pass5, axis=1, result_type="expand")
# Merge back into results_df
results_df["major_role_group"] = refined_records["major_role_group"]
results_df["reason_pass5"] = refined_records["reason_pass5"]
# Save a new CSV with the five-pass results
output_path_5 = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v754_five_pass.csv"
results_df.to_csv(output_path_5, index=False)
print(f"✅ Fifth pass complete. Saved refined results to: {output_path_5}")

🚀 Starting Fifth-Pass refinement for Coach / Coordinator / Manager...
✅ Fifth pass complete. Saved refined results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs/Job_Classifications_Batch_gpt4o_v754_five_pass.csv


# ==== 8) Generate Summary Statistics and Examples ====

In [ ]:
# Load the results
preds = results_df.copy()
# Generate summary statistics
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()
# Create summary
summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles', 'specialist_count', 'executive_lead_count', 'technician_count', 'skilled_laborer_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum()),
        int((preds['major_role_group'] == 'Technician').sum()),
        int((preds['major_role_group'] == 'Skilled Laborer').sum())
    ]
})
summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v755_five_pass.csv"
summary_stats.to_csv(summary_path, index=False)
# Show examples of classifications
examples = preds[['source_row_index', 'job_title_original', 'new_job_title',
                  'major_role_group', 'minor_sub_group']].head(10)
examples_path = OUTPUTS_DIR / "examples_gpt4o_v755_five_pass.csv"
examples.to_csv(examples_path, index=False)
print("\n📊 Summary Statistics:")
print(summary_stats.to_string(index=False))
print("\n📝 Major Role Distribution:")
print(major_counts.to_string())
print("\n📝 Minor Role Distribution:")
print(minor_counts.to_string())
print("\n📝 Example Classifications:")
print(examples.to_string(index=False))
print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")


📊 Summary Statistics:
               metric  value
           total_rows    102
   unique_major_roles     25
   unique_minor_roles      4
     specialist_count      2
 executive_lead_count      0
     technician_count      5
skilled_laborer_count      2

📝 Major Role Distribution:
major_role_group
Coordinator                       18
Manager                           17
Teacher                           16
Analyst                            8
Technician                         5
Coach                              5
Director                           5
Administrative Assistant           5
Architect (Technology-Focused)     3
Specialist                         2
Skilled Laborer                    2
Accountant                         2
Architect (Facility-Focused)       2
Principal                          1
Auditor                            1
Foreman                            1
Technologist                       1
Intern                             1
Developer                         

# ==== 9) Enhanced Quality Check and Validation ====

In [ ]:
# Check for alignment issues between justification and selected roles
alignment_issues = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()
    # Check if justification mentions the selected role
    if major_role not in justification and major_role != 'other':
        alignment_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': row['major_role_group'],
            'justification_excerpt': row['grouping_justification'][:100] + '...'
        })
# Check for job title format consistency
title_format_issues = []
for idx, row in preds.iterrows():
    new_title = str(row['new_job_title'])
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    # Check if job title incorporates both major and minor roles
    if major_role.lower() not in new_title.lower() or minor_role.lower() not in new_title.lower():
        title_format_issues.append({
            'row_index': row['source_row_index'],
            'new_job_title': new_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role
        })
# Check for executive roles with "Lead" minor sub-grouping (should be rare)
executive_lead_issues = []
for idx, row in preds.iterrows():
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    if major_role in EXECUTIVE_ROLES and minor_role == 'Lead':
        executive_lead_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'new_job_title': row['new_job_title']
        })
# Save quality check results
if alignment_issues:
    alignment_df = pd.DataFrame(alignment_issues)
    alignment_path = OUTPUTS_DIR / "alignment_issues_gpt4o_v755_four_pass.csv"
    alignment_df.to_csv(alignment_path, index=False)
    print(f"⚠️  Found {len(alignment_issues)} alignment issues - saved to {alignment_path}")
else:
    print("✅ No alignment issues found")
if title_format_issues:
    title_format_df = pd.DataFrame(title_format_issues)
    title_format_path = OUTPUTS_DIR / "title_format_issues_gpt4o_v755_four_pass.csv"
    title_format_df.to_csv(title_format_path, index=False)
    print(f"⚠️  Found {len(title_format_issues)} title format issues - saved to {title_format_path}")
else:
    print("✅ No title format issues found")
if executive_lead_issues:
    executive_lead_df = pd.DataFrame(executive_lead_issues)
    executive_lead_path = OUTPUTS_DIR / "executive_lead_issues_gpt4o_v755_four_pass.csv"
    executive_lead_df.to_csv(executive_lead_path, index=False)
    print(f"⚠️  Found {len(executive_lead_issues)} executive roles with 'Lead' minor sub-grouping - saved to {executive_lead_path}")
else:
    print("✅ No executive roles with inappropriate 'Lead' minor sub-grouping found")
print("\n✅ Enhanced quality check completed")

⚠️  Found 12 alignment issues - saved to /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs/alignment_issues_gpt4o_v755_four_pass.csv
⚠️  Found 16 title format issues - saved to /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs/title_format_issues_gpt4o_v755_four_pass.csv
✅ No executive roles with inappropriate 'Lead' minor sub-grouping found

✅ Enhanced quality check completed


# Post-Processing Validation and Corrections
**Purpose:** Fix known classification errors using deterministic rules

**What this does:**
1. Fixes Teacher/Principal/Assistant Principal/Counselor/Director sub-grouping errors
2. Flags and optionally corrects Manager/Coordinator misclassifications
3. Generates validation reports

**Expected impact:** Improves accuracy by fixing systematic errors

# ==== 10) Post-Processing: Fix Special Role Sub-Groupings ====

In [ ]:
import re
def fix_special_role_subgroups(df):
    """
    Fix the Teacher/Principal/Assistant Principal/Counselor/Director sub-grouping issue.
    These roles should not have I/II/III sub-groupings (only 'Lead' occasionally).
    """
    NO_SUBGROUP_ROLES = ['Teacher', 'Principal', 'Assistant Principal', 'Counselor', 'Director']
    FORBIDDEN_SUBGROUPS = ['I', 'II', 'III', '1', '2', '3']
    fixes_made = []
    for idx, row in df.iterrows():
        major_role = str(row['major_role_group']).strip()
        minor_sub = str(row['minor_sub_group']).strip()
        # Check if this is a special role with forbidden sub-grouping
        if major_role in NO_SUBGROUP_ROLES and minor_sub in FORBIDDEN_SUBGROUPS:
            # Fix it
            old_minor = minor_sub
            df.at[idx, 'minor_sub_group'] = ''
            # Update the new_job_title to remove the sub-group
            old_title = str(row['new_job_title'])
            # Remove the sub-group suffix (I, II, III)
            new_title = re.sub(r'\s+(I{1,3}|[123])$', '', old_title)
            df.at[idx, 'new_job_title'] = new_title
            fixes_made.append({
                'row': idx,
                'job_title_original': row.get('job_title_original', 'N/A'),
                'major_role': major_role,
                'old_minor': old_minor,
                'old_title': old_title,
                'new_title': new_title
            })
        # Special check for Director - should NEVER have any sub-grouping
        if major_role == 'Director' and minor_sub not in ['', 'nan', 'None', 'NaN']:
            if minor_sub not in FORBIDDEN_SUBGROUPS:  # Catch even 'Lead' for Director
                old_minor = minor_sub
                df.at[idx, 'minor_sub_group'] = ''
                old_title = str(row['new_job_title'])
                # Remove any sub-group suffix including 'Lead'
                new_title = re.sub(r'\s+(Lead|I{1,3}|[123])$', '', old_title)
                df.at[idx, 'new_job_title'] = new_title
                fixes_made.append({
                    'row': idx,
                    'job_title_original': row.get('job_title_original', 'N/A'),
                    'major_role': major_role,
                    'old_minor': old_minor,
                    'old_title': old_title,
                    'new_title': new_title
                })
    if fixes_made:
        print(f"✓ Fixed {len(fixes_made)} special role sub-grouping errors:")
        for fix in fixes_made:
            print(f"  Row {fix['row']}: {fix['major_role']} {fix['old_minor']} → {fix['major_role']}")
            print(f"    Title: '{fix['old_title']}' → '{fix['new_title']}'")
    else:
        print("✓ No special role sub-grouping errors found")
    return df, fixes_made
print("✅ Post-processing function 1 loaded: fix_special_role_subgroups()")

✅ Post-processing function 1 loaded: fix_special_role_subgroups()


# ==== 11) Post-Processing: Validate Manager/Coordinator Classifications ====

In [ ]:
def validate_manager_coordinator(df, confidence_threshold=3):
    """
    Identify Manager/Coordinator classifications that may be incorrect based on
    justification analysis. Returns a report, doesn't modify data.
    """
    issues = []
    for idx, row in df.iterrows():
        major_role = str(row['major_role_group']).strip()
        justification = str(row.get('grouping_justification', '')).lower()
        if major_role not in ['Manager', 'Coordinator']:
            continue
        # Signals that suggest Manager
        manager_signals = {
            'master_degree': ('master' in justification and 'degree' in justification),
            'admin_license': ('administrative license' in justification or
                            'admin license' in justification),
            'supervision': ('supervise' in justification or 'supervision' in justification),
            'budget': 'budget' in justification,
            'strategic': ('strategic' in justification and 'plan' in justification),
            'leads': ('leads implementation' in justification or
                     'leading implementation' in justification),
            'manages': 'manages' in justification or 'management' in justification
        }
        # Signals that suggest Coordinator
        coordinator_signals = {
            'bachelor_preferred': ('bachelor' in justification and 'preferred' in justification),
            'facilitates': 'facilitates' in justification or 'facilitation' in justification,
            'liaison': 'liaison' in justification,
            'coordinates': 'coordinates' in justification or 'coordination' in justification,
            'documentation': ('documentation' in justification and 'strategic' not in justification),
            'collaboration': ('collaboration' in justification and 'supervise' not in justification),
            'no_supervision': 'supervise' not in justification
        }
        manager_score = sum(manager_signals.values())
        coordinator_score = sum(coordinator_signals.values())
        # Identify potential misclassifications
        if major_role == 'Coordinator' and manager_score >= confidence_threshold:
            issues.append({
                'row': idx,
                'job_title_original': row.get('job_title_original', 'N/A'),
                'current': 'Coordinator',
                'suggested': 'Manager',
                'manager_score': manager_score,
                'coordinator_score': coordinator_score,
                'confidence': 'HIGH' if manager_score >= confidence_threshold + 2 else 'MEDIUM'
            })
        elif major_role == 'Manager' and coordinator_score >= confidence_threshold:
            issues.append({
                'row': idx,
                'job_title_original': row.get('job_title_original', 'N/A'),
                'current': 'Manager',
                'suggested': 'Coordinator',
                'manager_score': manager_score,
                'coordinator_score': coordinator_score,
                'confidence': 'HIGH' if coordinator_score >= confidence_threshold + 2 else 'MEDIUM'
            })
    if issues:
        print(f"\n⚠️  Found {len(issues)} potential Manager/Coordinator misclassifications:")
        for issue in issues:
            print(f"\n  Row {issue['row']}: {issue['job_title_original']}")
            print(f"    Current: {issue['current']}")
            print(f"    Suggested: {issue['suggested']} (confidence: {issue['confidence']})")
            print(f"    Scores: Manager={issue['manager_score']}, Coordinator={issue['coordinator_score']}")
    else:
        print("✓ No obvious Manager/Coordinator misclassifications detected")
    return issues
print("✅ Post-processing function 2 loaded: validate_manager_coordinator()")

✅ Post-processing function 2 loaded: validate_manager_coordinator()


# ==== 12) Post-Processing: Apply Manager/Coordinator Corrections (Optional) ====

In [ ]:
def apply_manager_coordinator_corrections(df, confidence_threshold=4):
    """
    Automatically apply Manager/Coordinator corrections for HIGH confidence cases only.
    Use confidence_threshold=4 or higher to be conservative.
    """
    corrections_made = []
    for idx, row in df.iterrows():
        major_role = str(row['major_role_group']).strip()
        justification = str(row.get('grouping_justification', '')).lower()
        if major_role not in ['Manager', 'Coordinator']:
            continue
        # Same signal detection as validate function
        manager_signals = {
            'master_degree': ('master' in justification and 'degree' in justification),
            'admin_license': ('administrative license' in justification or
                            'admin license' in justification),
            'supervision': ('supervise' in justification or 'supervision' in justification),
            'budget': 'budget' in justification,
            'strategic': ('strategic' in justification and 'plan' in justification),
            'leads': ('leads implementation' in justification or
                     'leading implementation' in justification),
            'manages': 'manages' in justification or 'management' in justification
        }
        coordinator_signals = {
            'bachelor_preferred': ('bachelor' in justification and 'preferred' in justification),
            'facilitates': 'facilitates' in justification or 'facilitation' in justification,
            'liaison': 'liaison' in justification,
            'coordinates': 'coordinates' in justification or 'coordination' in justification,
            'documentation': ('documentation' in justification and 'strategic' not in justification),
            'collaboration': ('collaboration' in justification and 'supervise' not in justification)
        }
        manager_score = sum(manager_signals.values())
        coordinator_score = sum(coordinator_signals.values())
        # Only apply corrections for very high confidence cases
        if major_role == 'Coordinator' and manager_score >= confidence_threshold:
            old_role = major_role
            df.at[idx, 'major_role_group'] = 'Manager'
            # Update job title
            old_title = str(row['new_job_title'])
            new_title = old_title.replace('Coordinator', 'Manager')
            df.at[idx, 'new_job_title'] = new_title
            corrections_made.append({
                'row': idx,
                'job_title_original': row.get('job_title_original', 'N/A'),
                'old_role': old_role,
                'new_role': 'Manager',
                'score': manager_score
            })
        elif major_role == 'Manager' and coordinator_score >= confidence_threshold:
            old_role = major_role
            df.at[idx, 'major_role_group'] = 'Coordinator'
            # Update job title
            old_title = str(row['new_job_title'])
            new_title = old_title.replace('Manager', 'Coordinator')
            df.at[idx, 'new_job_title'] = new_title
            corrections_made.append({
                'row': idx,
                'job_title_original': row.get('job_title_original', 'N/A'),
                'old_role': old_role,
                'new_role': 'Coordinator',
                'score': coordinator_score
            })
    if corrections_made:
        print(f"\n✓ Applied {len(corrections_made)} high-confidence Manager/Coordinator corrections:")
        for corr in corrections_made:
            print(f"  Row {corr['row']}: {corr['old_role']} → {corr['new_role']} (score: {corr['score']})")
    else:
        print("\n✓ No high-confidence Manager/Coordinator corrections needed")
    return df, corrections_made
print("✅ Post-processing function 3 loaded: apply_manager_coordinator_corrections()")
# NEW POST-PROCESSING FUNCTION: Correct Specialist Misclassifications
def correct_specialist_misclassifications(df, confidence_threshold=2):
    """
    Automatically correct 'Specialist' roles that clearly belong to another category.
    Only applies corrections when multiple keywords strongly indicate a different role.
    """
    corrections_made = []
    ROLE_INDICATORS = {
        'Teacher': ['teach', 'instruction', 'lesson', 'curriculum', 'classroom'],
        'Coach': ['coach', 'mentor', 'professional development', 'co-teach', 'PLC'],
        'Analyst': ['analyze', 'data analysis', 'evaluate', 'metrics', 'reporting'],
        'Technician': ['repair', 'troubleshoot', 'install', 'technical support'],
        'Accountant': ['accounting', 'payroll', 'bookkeeping', 'audit'],
        'Coordinator': ['coordinate', 'liaison', 'organize', 'facilitate']
    }

    for idx, row in df.iterrows():
        if row['major_role_group'] != 'Specialist':
            continue

        full_text = (str(row['grouping_justification']) + " " +
                     str(row.get('Position Summary', ''))).lower()

        for target_role, keywords in ROLE_INDICATORS.items():
            matches = sum(1 for kw in keywords if kw in full_text)
            if matches >= confidence_threshold:
                # Apply correction
                old_title = row['new_job_title']
                new_title = old_title.replace('Specialist', target_role)

                df.at[idx, 'major_role_group'] = target_role
                df.at[idx, 'new_job_title'] = new_title

                corrections_made.append({
                    'row': idx,
                    'old_role': 'Specialist',
                    'new_role': target_role,
                    'keywords_matched': matches
                })
                break  # Only apply one correction

    if corrections_made:
        print(f"\n✓ Corrected {len(corrections_made)} 'Specialist' misclassifications:")
        for corr in corrections_made:
            print(f"  Row {corr['row']}: Specialist → {corr['new_role']} (keywords matched: {corr['keywords_matched']})")

    return df, corrections_made

print("✅ Post-processing function 4 loaded: correct_specialist_misclassifications()")

✅ Post-processing function 3 loaded: apply_manager_coordinator_corrections()
✅ Post-processing function 4 loaded: correct_specialist_misclassifications()


# ==== 11B) Post-Processing: Driver and Clerk/Admin Corrections ====

In [ ]:
def correct_driver_vs_skilled_laborer(df, confidence_threshold=2):
    """
    Correct Driver vs Skilled Laborer misclassifications.

    Rules:
    - If role focuses on "operating vehicles" AND "requires a CDL",
      it should be classified as "Driver", not "Skilled Laborer"
    """
    corrections_made = []

    for idx, row in df.iterrows():
        major_role = str(row['major_role_group']).strip()
        if major_role != 'Skilled Laborer':
            continue

        justification = str(row.get('grouping_justification', '')).lower()
        summary = str(row.get('Position Summary', '')).lower()
        full_text = justification + " " + summary

        driver_signals = {
            'operates_vehicle': any(phrase in full_text for phrase in [
                'operate vehicle', 'operates vehicle', 'operating vehicle',
                'drive', 'driving', 'driver'
            ]),
            'requires_cdl': any(phrase in full_text for phrase in [
                'cdl', 'commercial driver', 'commercial driving license',
                'class a license', 'class b license'
            ]),
            'transport': any(phrase in full_text for phrase in [
                'transport', 'transportation', 'route', 'bus'
            ]),
            'vehicle_focus': any(phrase in full_text for phrase in [
                'vehicle maintenance', 'vehicle operation', 'vehicle safety',
                'fleet', 'driving duties'
            ])
        }

        driver_score = sum(driver_signals.values())
        has_critical_combo = driver_signals['operates_vehicle'] and driver_signals['requires_cdl']

        if has_critical_combo or driver_score >= confidence_threshold:
            old_role = major_role
            df.at[idx, 'major_role_group'] = 'Driver'

            # Fix title - construct clean title to avoid duplication
            old_title = str(row['new_job_title'])
            minor_sub = str(row.get('minor_sub_group', '')).strip()

            # Build new title - only use 'Driver' once
            if minor_sub:
                new_title = f'Driver {minor_sub}'
            else:
                new_title = 'Driver'

            df.at[idx, 'new_job_title'] = new_title

            corrections_made.append({
                'row': idx,
                'job_title_original': row.get('job_title_original', 'N/A'),
                'old_role': old_role,
                'old_title': old_title,
                'new_role': 'Driver',
                'new_title': new_title,
                'driver_score': driver_score,
                'critical_combo': has_critical_combo,
                'reason': 'Operates vehicles + CDL required' if has_critical_combo else f'Driver score: {driver_score}'
            })

    if corrections_made:
        print(f"\n✓ Corrected {len(corrections_made)} Driver vs Skilled Laborer misclassifications:")
        for corr in corrections_made:
            print(f"  Row {corr['row']}: {corr['old_role']} → {corr['new_role']}")
            print(f"    Title: '{corr['old_title']}' → '{corr['new_title']}'")
            print(f"    Reason: {corr['reason']}")
    else:
        print("\n✓ No Driver vs Skilled Laborer corrections needed")

    return df, corrections_made

print("✅ Post-processing function 5 loaded: correct_driver_vs_skilled_laborer()")


✅ Post-processing function 5 loaded: correct_driver_vs_skilled_laborer()


# ==== 13) Run Post-Processing Workflow ====

In [ ]:
print("="*80)
print("POST-PROCESSING CLASSIFICATION RESULTS")
print("="*80)
# Make a copy to preserve original
preds_corrected = preds.copy()

# Step 1: Fix special role sub-grouping issues (automatic)
print("\nStep 1: Fixing special role sub-groupings...")
preds_corrected, subgroup_fixes = fix_special_role_subgroups(preds_corrected)

# Step 2: Validate Manager/Coordinator (report only)
print("\nStep 2: Validating Manager/Coordinator classifications...")
manager_coord_issues = validate_manager_coordinator(preds_corrected, confidence_threshold=3)

# Step 3: Apply high-confidence Manager/Coordinator corrections (ENABLED)
APPLY_MANAGER_COORDINATOR_CORRECTIONS = True  # Enable automatic corrections
if APPLY_MANAGER_COORDINATOR_CORRECTIONS:
    print("\nStep 3: Applying high-confidence Manager/Coordinator corrections...")
    preds_corrected, corrections = apply_manager_coordinator_corrections(preds_corrected, confidence_threshold=4)
else:
    print("\nStep 3: Skipped automatic Manager/Coordinator corrections (APPLY_MANAGER_COORDINATOR_CORRECTIONS=False)")
    corrections = []

# ==== NEW: Restore valid Specialist roles that were incorrectly downgraded ====
def restore_valid_specialists(df):
    """Restore 'Specialist' classification for roles that were incorrectly downgraded."""
    corrections = []

    for idx, row in df.iterrows():
        current_role = row['major_role_group']
        original_title = str(row.get('job_title_original', '')).lower()
        summary = str(row.get('Position Summary', '')).lower()

        # If it was likely a Specialist but got downgraded
        if 'specialist' in original_title and current_role != 'Specialist':
            # Check if it's NOT truly a Coach/Analyst/Manager
            is_coach = 'instructional' in summary or 'teacher development' in summary
            is_analyst = 'statistical' in summary or 'data analysis' in summary
            is_manager = 'supervise' in summary or 'budget' in summary

            if not (is_coach or is_analyst or is_manager):
                # Restore to Specialist
                df.at[idx, 'major_role_group'] = 'Specialist'
                old_title = row['new_job_title']
                new_title = old_title.replace(current_role, 'Specialist')
                df.at[idx, 'new_job_title'] = new_title
                corrections.append({
                    'row': idx,
                    'original_title': original_title,
                    'restored_to': 'Specialist'
                })

    if corrections:
        print(f"\n✅ Restored {len(corrections)} roles to 'Specialist'")
        for c in corrections:
            print(f"  Row {c['row']}: {c['original_title']}")

    return df, corrections

# CALL THE FUNCTION HERE — THIS IS THE CRITICAL STEP YOU MISSED
print("\nStep 4: Restoring valid 'Specialist' roles incorrectly downgraded...")
preds_corrected, specialist_restores = restore_valid_specialists(preds_corrected)

# Step 5: Optionally correct Specialist misclassifications (DISABLED for now)
APPLY_SPECIALIST_CORRECTIONS = False  # Set to True to enable
if APPLY_SPECIALIST_CORRECTIONS:
    print("\nStep 5: Applying high-confidence Specialist corrections...")
    preds_corrected, specialist_corrections = correct_specialist_misclassifications(preds_corrected, confidence_threshold=2)
else:
    print("\nStep 5: Skipped automatic Specialist corrections (APPLY_SPECIALIST_CORRECTIONS=False)")
    specialist_corrections = []

# Step 6: Correct Driver vs Skilled Laborer misclassifications
APPLY_DRIVER_CORRECTIONS = True  # Set to False to disable
if APPLY_DRIVER_CORRECTIONS:
    print("\nStep 6: Correcting Driver vs Skilled Laborer misclassifications...")
    preds_corrected, driver_corrections = correct_driver_vs_skilled_laborer(
        preds_corrected,
        confidence_threshold=2
    )
else:
    print("\nStep 6: Skipped Driver corrections (APPLY_DRIVER_CORRECTIONS=False)")
    driver_corrections = []

print("\n" + "="*80)
print("POST-PROCESSING SUMMARY")
print("="*80)
print(f"✓ Special role sub-grouping fixes applied: {len(subgroup_fixes)}")
print(f"✓ Manager/Coordinator issues flagged for review: {len(manager_coord_issues)}")
if APPLY_MANAGER_COORDINATOR_CORRECTIONS:
    print(f"✓ Manager/Coordinator corrections applied: {len(corrections)}")
else:
    print(f"ℹ️  Manager/Coordinator automatic corrections disabled")
    print(f"   To enable: Set APPLY_MANAGER_COORDINATOR_CORRECTIONS = True above")
if APPLY_SPECIALIST_CORRECTIONS:
    print(f"✓ Specialist corrections applied: {len(specialist_corrections)}")
else:
    print(f"ℹ️  Specialist automatic corrections disabled")
    print(f"   To enable: Set APPLY_SPECIALIST_CORRECTIONS = True above")
if APPLY_DRIVER_CORRECTIONS:
    print(f"✓ Driver vs Skilled Laborer corrections applied: {len(driver_corrections)}")
else:
    print(f"ℹ️  Driver corrections disabled")
    print(f"   To enable: Set APPLY_DRIVER_CORRECTIONS = True above")


POST-PROCESSING CLASSIFICATION RESULTS

Step 1: Fixing special role sub-groupings...
✓ Fixed 22 special role sub-grouping errors:
  Row 0: Teacher I → Teacher
    Title: 'Teacher I' → 'Teacher'
  Row 2: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Row 5: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Row 6: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Row 10: Principal II → Principal
    Title: 'Principal II' → 'Principal'
  Row 11: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Row 13: Director III → Director
    Title: 'Director of Pre-Kindergarten Programs' → 'Director of Pre-Kindergarten Programs'
  Row 29: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Row 35: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Row 36: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Row 43: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Row 44: Teacher II → Teacher
    Title: 'Teacher II' → 'Teacher'
  Ro

In [ ]:
# ==== 13B NEW: Validate Specialist Classifications ====
def validate_specialist_roles(df):
    """Review all 'Specialist' classifications and flag those that should be something else."""
    specialist_issues = []
    for idx, row in df.iterrows():
        if row['major_role_group'] != 'Specialist':
            continue

        full_text = (str(row['grouping_justification']) + " " +
                    str(row.get('Position Summary', ''))).lower()

        # Flag if it should be Teacher
        if 'teach' in full_text or 'instruction' in full_text or 'lesson' in full_text:
            if 'O&M' in str(row.get('job_title_original', '')) or 'Orientation and Mobility' in full_text:
                # O&M Specialist is correctly a Teacher
                pass
            else:
                specialist_issues.append({
                    'row': idx,
                    'issue': 'Should be Teacher',
                    'context': 'Role involves teaching/instruction'
                })

        # Flag if it should be Coach
        elif any(kw in full_text for kw in ['coach', 'professional development', 'mentor']):
            specialist_issues.append({
                'row': idx,
                'issue': 'Should be Coach',
                'context': 'Role involves coaching staff'
            })

        # Flag if it should be Accountant
        elif any(kw in full_text for kw in ['accounting', 'payroll', 'audit']):
            specialist_issues.append({
                'row': idx,
                'issue': 'Should be Accountant',
                'context': 'Role involves financial/accounting tasks'
            })

    return specialist_issues

# Run the validation
print("\n🔍 Validating Specialist roles...")
specialist_flags = validate_specialist_roles(preds_corrected)
if specialist_flags:
    print(f"⚠️  Found {len(specialist_flags)} Specialist roles that may be incorrect:")
    for flag in specialist_flags:
        print(f"  Row {flag['row']}: {flag['issue']} ({flag['context']})")
else:
    print("✅ All Specialist roles appear correct.")


🔍 Validating Specialist roles...
⚠️  Found 1 Specialist roles that may be incorrect:
  Row 96: Should be Teacher (Role involves teaching/instruction)


# ==== 14) Save Post-Processed Results ====

In [ ]:
# Save the corrected results
output_path_corrected = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v757_five_pass_CORRECTED.csv"
preds_corrected.to_csv(output_path_corrected, index=False)
print(f"\n✅ Saved post-processed results to: {output_path_corrected}")
# Save a report of fixes made
if subgroup_fixes:
    fixes_df = pd.DataFrame(subgroup_fixes)
    fixes_path = OUTPUTS_DIR / "post_processing_subgroup_fixes_v757.csv"
    fixes_df.to_csv(fixes_path, index=False)
    print(f"✅ Saved sub-grouping fixes report to: {fixes_path}")
if manager_coord_issues:
    issues_df = pd.DataFrame(manager_coord_issues)
    issues_path = OUTPUTS_DIR / "post_processing_manager_coordinator_flagged_v757.csv"
    issues_df.to_csv(issues_path, index=False)
    print(f"✅ Saved Manager/Coordinator flagged issues to: {issues_path}")
if specialist_corrections:
    corrections_df = pd.DataFrame(specialist_corrections)
    corrections_path = OUTPUTS_DIR / "post_processing_specialist_corrections_v757.csv"
    corrections_df.to_csv(corrections_path, index=False)
    print(f"✅ Saved Specialist corrections report to: {corrections_path}")

# Save Driver corrections report
if driver_corrections:
    driver_report_df = pd.DataFrame(driver_corrections)
    driver_report_path = OUTPUTS_DIR / "post_processing_driver_corrections_v757.csv"
    driver_report_df.to_csv(driver_report_path, index=False)
    print(f"✅ Saved Driver corrections report to: {driver_report_path}")




✅ Saved post-processed results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs/Job_Classifications_Batch_gpt4o_v757_five_pass_CORRECTED.csv
✅ Saved sub-grouping fixes report to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs/post_processing_subgroup_fixes_v757.csv
✅ Saved Manager/Coordinator flagged issues to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs/post_processing_manager_coordinator_flagged_v757.csv
✅ Saved Driver corrections report to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251202_173326/outputs/post_processing_driver_corrections_v757.csv


# ==== 15) Optional: Compare Before and After Post-Processing ====

In [ ]:
# Compare original vs corrected to see what changed
comparison = pd.DataFrame({
    'source_row_index': preds['source_row_index'],
    'job_title_original': preds['job_title_original'],
    'original_major': preds['major_role_group'],
    'original_minor': preds['minor_sub_group'],
    'original_title': preds['new_job_title'],
    'corrected_major': preds_corrected['major_role_group'],
    'corrected_minor': preds_corrected['minor_sub_group'],
    'corrected_title': preds_corrected['new_job_title']
})
# Find rows where corrections were made
major_changed = comparison['original_major'] != comparison['corrected_major']
minor_changed = comparison['original_minor'] != comparison['corrected_minor']
title_changed = comparison['original_title'] != comparison['corrected_title']
any_changed = major_changed | minor_changed | title_changed
if any_changed.sum() > 0:
    print(f"\n📊 Total changes made: {any_changed.sum()} rows")
    print(f"  - Major role changed: {major_changed.sum()}")
    print(f"  - Minor sub-group changed: {minor_changed.sum()}")
    print(f"  - Job title changed: {title_changed.sum()}")
    print("\nRows with changes:")
    print(comparison[any_changed][['source_row_index', 'job_title_original',
                                    'original_major', 'original_minor',
                                    'corrected_major', 'corrected_minor']].to_string())
    # Save comparison
    comparison_path = OUTPUTS_DIR / "post_processing_comparison_v757.csv"
    comparison[any_changed].to_csv(comparison_path, index=False)
    print(f"\n✅ Saved comparison report to: {comparison_path}")
else:
    print("\nℹ️  No changes were made (all classifications were already correct)")


📊 Total changes made: 26 rows
  - Major role changed: 4
  - Minor sub-group changed: 22
  - Job title changed: 21

Rows with changes:
    source_row_index job_title_original   original_major original_minor corrected_major corrected_minor
0                  0                             Teacher              I         Teacher                
2                  2                             Teacher             II         Teacher                
5                  5                             Teacher             II         Teacher                
6                  6                             Teacher             II         Teacher                
7                  7                     Skilled Laborer              I          Driver               I
10                10                           Principal             II       Principal                
11                11                             Teacher             II         Teacher                
13                13             